In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
n_samples = 500

age = np.random.normal(35, 10, n_samples).clip(18, 70)
income = np.random.normal(50000, 15000, n_samples).clip(10000, 150000)
visits = np.random.poisson(5, n_samples)
session_duration = np.random.normal(8, 3, n_samples).clip(0.5, 30)

income_scaled = income / 50000
visits_scaled = visits / 5
interaction = income_scaled * visits_scaled

z = (
    0.05 * visits +
    0.03 * session_duration -
    0.02 * age -
    3.0 +
    1.5 * interaction
)
probability = 1 / (1 + np.exp(-z))
purchased = np.random.binomial(1, probability)

data = pd.DataFrame({
    'age': age, 'income': income, 'visits': visits,
    'session_duration': session_duration, 'purchased': purchased
})

X = data[['age', 'income', 'visits', 'session_duration']]
y = data['purchased']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
tf.random.set_seed(42)
model = Sequential()
model.add(Dense(units=25, activation='sigmoid', input_shape=(4,)))
model.add(Dense(units=15, activation='sigmoid'))
model.add(Dense(units=1, activation='sigmoid'))
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X_train_scaled, y_train, epochs=400, validation_split=0.1, verbose=0)

print("Model retrained. Test accuracy:", model.evaluate(X_test_scaled, y_test, verbose=0)[1])
weights = model.get_weights()
len(weights)
for i, w in enumerate(weights):
    print(f"weights[{i}] shape: {w.shape}")


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model retrained. Test accuracy: 0.7400000095367432


6

In [3]:
for i, w in enumerate(weights):
    print(f"weights[{i}] shape: {w.shape}")

weights[0] shape: (4, 25)
weights[1] shape: (25,)
weights[2] shape: (25, 15)
weights[3] shape: (15,)
weights[4] shape: (15, 1)
weights[5] shape: (1,)


In [8]:
def dense_layer(X,W,b):
  Z=X @ W +b
  A=1/(1+np.exp(-Z))
  return A

def forward_prop(X,weights):
  W1,b1,W2,b2,W3,b3=weights
  A1=dense_layer(X,W1,b1)
  A2=dense_layer(A1,W2,b2)
  A3=dense_layer(A2,W3,b3)
  return A3

In [9]:
sample_customer = X_test_scaled[0:1]
our_prediction = forward_prop(sample_customer, weights)
print("Our NumPy prediction:", our_prediction)

keras_prediction = model.predict(sample_customer, verbose=0)
print("Keras prediction:", keras_prediction)

Our NumPy prediction: [[0.08104385]]
Keras prediction: [[0.08104385]]


In [11]:
our_predictions = forward_prop(X_test_scaled, weights)
keras_predictions = model.predict(X_test_scaled, verbose=0)

max_difference = np.max(np.abs(our_predictions - keras_predictions))
print("Maximum difference across all test customers:", max_difference)

Maximum difference across all test customers: 7.621918080014112e-08


In [12]:
our_labels =(our_predictions > 0.5).astype(int)
our_labels=our_labels.flatten()
our_accuracy=np.mean(our_labels == y_test.values)
print("Accuracy using from-scratch predictions:",our_accuracy)

ACcuracy using from-scratch predictions: 0.74
